# Qwen3-30B-A3B on one 40 GiB GPU

The official pre-quantized checkpoint contains Qwen3-30B-A3B in GPTQ Int4. 
Using it avoids the temporary 39+ GiB peak caused by on-the-fly BitsAndBytes quantization.

Also supported by transformerlens as Architecture Qwen3MoeForCausalLM

In [1]:
import gc
import os
import subprocess
import sys
from pathlib import Path

import torch
from dotenv import load_dotenv

load_dotenv("/glazkov-dev/.env", override=True)
os.environ.setdefault("HF_HUB_CACHE", "/glazkov-dev/.cache")
os.environ["PATH"] = f"{Path(sys.executable).parent}:{os.environ['PATH']}"
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is missing from /glazkov-dev/.env"

# Hugging Face cache paths are captured while GPTQModel is imported.
from gptqmodel import GPTQModel

gc.collect()


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


WARN  Feature `utils/Perplexity` requires Python < 3.14 and Python GIL enabled and Python >= 3.13.3T (T for Threading-Free edition of Python) plus Torch 2.8. Feature is currently skipped/disabled.


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [2]:
source_model_name = "Qwen/Qwen3-30B-A3B"
model_name = "Qwen/Qwen3-30B-A3B-GPTQ-Int4"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required")

# Query VRAM without creating CUDA contexts on other, possibly full GPUs.
gpu_rows = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=index,memory.total,memory.free",
        "--format=csv,noheader,nounits",
    ],
    text=True,
).strip().splitlines()
gpu_memory = []
for row in gpu_rows:
    index, total_mib, free_mib = (int(value.strip()) for value in row.split(","))
    gpu_memory.append((index, total_mib, free_mib))
    print(f"cuda:{index}: {free_mib / 1024:.1f} / {total_mib / 1024:.1f} GiB free")

gpu_id, _, free_mib = max(gpu_memory, key=lambda item: item[2])
free_gib = free_mib / 1024
if free_gib < 22:
    raise RuntimeError(
        f"The freest GPU is cuda:{gpu_id}, but it has only {free_gib:.1f} GiB free. "
        "Free at least 22 GiB or restart this kernel."
    )
print(f"Loading on cuda:{gpu_id}")
with torch.cuda.device(gpu_id):
    torch.cuda.empty_cache()

model = GPTQModel.load(
    model_name,
    device=f"cuda:{gpu_id}",
    profile="low_memory",
)
tokenizer = model.tokenizer
model.model.eval()
# Release temporary shard/conversion buffers retained by the CUDA allocator.
gc.collect()
with torch.cuda.device(gpu_id):
    torch.cuda.empty_cache()

allocated_gib = torch.cuda.memory_allocated(gpu_id) / 2**30
reserved_gib = torch.cuda.memory_reserved(gpu_id) / 2**30
print(f"Loaded: {allocated_gib:.1f} GiB allocated, {reserved_gib:.1f} GiB reserved")


cuda:0: 28.7 / 40.0 GiB free
cuda:1: 0.3 / 40.0 GiB free
cuda:2: 0.3 / 40.0 GiB free
cuda:3: 39.0 / 40.0 GiB free
Loading on cuda:3


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-30B-A3B-GPTQ-Int4/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


from_quantized: adapter: None


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/9b534e4318b7ebc3c961a839f13eb18b1833f441/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-30B-A3B-GPTQ-Int4/revision/main "HTTP/1.1 200 OK"


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

...

INFO  Model: Loaded `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}



INFO  Model: Auto-fixed `generation_config` mismatch between model and `generation_config.json`.


INFO  Model: Updated `generation_config`: GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "temperature": 0.6,
  "top_k": 20,
  "top_p": 0.95
}



INFO  Kernel: loaded -> `[MarlinLinear]`                                       


Loaded: 15.5 GiB allocated, 19.6 GiB reserved


In [3]:
prompt = "Give me a short introduction to large language models."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

input_device = next(model.model.parameters()).device
model_inputs = tokenizer(text, return_tensors="pt").to(input_device)
with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        use_cache=True,
    )

new_tokens = generated_ids[0, model_inputs.input_ids.shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


Large Language Models (LLMs) are advanced artificial intelligence systems trained on vast amounts of text data to understand and generate human-like language. They can perform a wide range of tasks, such as answering questions, writing essays, coding, and even engaging in conversation. These models are built using deep learning techniques, particularly transformer architectures, which allow them to capture complex patterns in language. LLMs like GPT, BERT, and others have revolutionized natural language processing and are widely used in applications ranging from virtual assistants to content creation.


In [4]:
model

Qwen3MoeQModel(
  (model): Qwen3MoeForCausalLM(
    (model): Qwen3MoeModel(
      (embed_tokens): Embedding(151936, 2048)
      (layers): ModuleList(
        (0-47): 48 x Qwen3MoeDecoderLayer(
          (self_attn): Qwen3MoeAttention(
            (q_proj): MarlinLinear()
            (k_proj): MarlinLinear()
            (v_proj): MarlinLinear()
            (o_proj): MarlinLinear()
            (q_norm): Qwen3MoeRMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3MoeRMSNorm((128,), eps=1e-06)
          )
          (mlp): LinearQwen3MoeSparseMoeBlock(
            (gate): Qwen3MoeTopKRouter()
            (experts): ModuleList(
              (0-127): 128 x Qwen3MoeMLP(
                (gate_proj): MarlinLinear()
                (up_proj): MarlinLinear()
                (down_proj): MarlinLinear()
                (act_fn): SiLUActivation()
              )
            )
          )
          (input_layernorm): Qwen3MoeRMSNorm((2048,), eps=1e-06)
          (post_attention_layernorm): Qwen3Moe

todo: read about rapid MarlinLinear (accelerate quantized linear kernels)

In [ ]:
# Run this before loading another model. Restarting the kernel is more reliable.
del model, model_inputs, generated_ids, new_tokens
gc.collect()
torch.cuda.empty_cache()
